In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# ==========================================
# 1. LOAD KAGGLE DATA
# ==========================================
print("Step 1: Fetching Kaggle Dataset...")

# Load the CSV
df = pd.read_csv('datasets/btcusd_1-min_data.csv')

# Convert the Unix 'Timestamp' column to a readable DateTime format
df['Date'] = pd.to_datetime(df['Timestamp'], unit='s')

# Drop the old Timestamp column and set Date as the index
df = df.drop(columns=['Timestamp'])
df.set_index('Date', inplace=True)

print(f"Loaded {df.shape[0]} raw 1-minute records.")
# ==========================================
# 2. PREPROCESSING: Convert Minutes to Days
# ==========================================
print("Step 2: Resampling and Engineering Features...")

# Resample 1-minute data into daily data.
df = df.resample('D').agg({
    'Open': 'first', 
    'High': 'max', 
    'Low': 'min', 
    'Close': 'last', 
    'Volume': 'sum'
})

# Forward-fill any days where the market data might be completely missing
df = df.ffill()

# --- Feature Engineering ---
df['Daily_Return'] = df['Close'].pct_change()
df['Volatility'] = (df['High'] - df['Low']) / df['Close']
df['SMA_7'] = df['Close'].rolling(window=7).mean()
df['SMA_30'] = df['Close'].rolling(window=30).mean()
df['Volume_Change'] = df['Volume'].pct_change()

# Create the Target Variable
df['Next_Day_Close'] = df['Close'].shift(-1)
df['Target'] = (df['Next_Day_Close'] > df['Close']).astype(int)

# --- THE FIX IS HERE ---
# Replace infinite values (created by division-by-zero on 0 volume/price days) with NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Now drop all rows containing NaN or newly converted infinite values
df.dropna(inplace=True)
# -----------------------

# Define our feature matrix (X) and target vector (y)
feature_cols = ['Daily_Return', 'Volatility', 'SMA_7', 'SMA_30', 'Volume_Change']
X = df[feature_cols]
y = df['Target']
# ==========================================
# 3. CHRONOLOGICAL TRAIN-TEST SPLIT
# ==========================================
print("Step 3: Performing Chronological Train-Test Split...")

# Time-series data must NOT be shuffled. We train on the past, test on the future.
split_index = int(len(df) * 0.8) # 80% Train, 20% Test

X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

print(f"Training set size: {X_train.shape[0]} days")
print(f"Testing set size: {X_test.shape[0]} days\n")


# ==========================================
# 4. DATA SCALING
# ==========================================
print("Step 4: Scaling Features...")
scaler = StandardScaler()

# Fit on training data only to avoid data leakage
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


# ==========================================
# 5. MODEL SELECTION & TRAINING
# ==========================================
print("Step 5: Training Supervised Machine Learning Models...")

# Model A: Logistic Regression Baseline
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_scaled, y_train)

# Model B: Random Forest Classifier
rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
rf_clf.fit(X_train_scaled, y_train)


# ==========================================
# 6. EVALUATION & RESULTS INTERPRETATION
# ==========================================
print("Step 6: Evaluating Models on Unseen Test Data...\n")

def evaluate_model(model, X_test_data, y_true, model_name):
    predictions = model.predict(X_test_data)
    acc = accuracy_score(y_true, predictions)
    cm = confusion_matrix(y_true, predictions)
    
    print(f"=== {model_name} Results ===")
    print(f"Accuracy: {acc:.2%}")
    print("\nConfusion Matrix:")
    print(f"True Down: {cm[0][0]} | False Up:   {cm[0][1]}")
    print(f"False Down: {cm[1][0]} | True Up:    {cm[1][1]}")
    print("\nClassification Report:")
    print(classification_report(y_true, predictions, target_names=['Down/Flat', 'Up']))
    print("-" * 40 + "\n")

# Run evaluations
evaluate_model(log_reg, X_test_scaled, y_test, "Logistic Regression")
evaluate_model(rf_clf, X_test_scaled, y_test, "Random Forest Classifier")

# Feature Importance Interpretation
print("=== Feature Importances (Random Forest) ===")
importances = rf_clf.feature_importances_
for col, importance in zip(feature_cols, importances):
    print(f"{col}: {importance:.2%}")

Step 1: Fetching Kaggle Dataset...


FileNotFoundError: [Errno 2] No such file or directory: 'datasets/btcusd_1-min_data.csv'